# Especialista PLS — Fundamentos e decisão metodológica
**Grupo 1 | Série diária do Bitcoin**

## 1. O problema que o PLS resolve

As 51 features do projeto contêm variáveis fortemente relacionadas: lags e médias móveis do fechamento, medidas de volume, indicadores técnicos e variáveis de calendário. Em regressão linear comum, essa multicolinearidade pode tornar os coeficientes instáveis. O Partial Least Squares (PLS) reduz `X` a um conjunto menor de componentes latentes construídos levando em conta também o alvo `y`.

## 2. Intuição e formulação

O PLS procura direções em `X` que tenham covariância elevada com `y`. Em uma decomposição conceitual:

$$X = T P^T + E, \qquad y = T q + f$$

`T` contém os escores latentes, `P` os pesos de reconstrução de `X`, `q` relaciona os escores ao alvo, e `E` e `f` são resíduos. As direções são extraídas sequencialmente; cada nova componente explica uma parte adicional da relação entre preditores e alvo.

O algoritmo NIPALS é uma forma iterativa comum de estimar os pesos e escores. Na prática do projeto, `sklearn.cross_decomposition.PLSRegression` executa o ajuste; com `scale=True`, centraliza e escala internamente `X` e `y` em cada ajuste.

## 3. PLS versus PCA

- **PCA** é não supervisionado: escolhe direções que explicam variância de `X`, sem consultar `y`.
- **PLS** é supervisionado: escolhe componentes de `X` informativas para a covariância com `y`.
- PCA pode priorizar uma direção muito variável, mas pouco relacionada ao alvo; PLS pode priorizar uma direção de menor variância em `X` que seja preditiva para `y`.

## 4. Hiperparâmetro principal

`n_components` controla quantas direções latentes entram na regressão. Poucas componentes podem deixar estrutura preditiva de fora (subajuste); muitas podem incorporar ruído e reduzir a generalização. O limite válido depende do número de features, amostras e do posto numérico dos dados.

No pipeline atual, a busca avaliou de 1 a 25 componentes usando cinco folds cronológicos e MAE. A escolha foi **9 componentes**, com MAE_CV de aproximadamente **USD 1.982,29**. O R²_CV não foi o critério de seleção: a configuração com menor MAE é que foi escolhida.

## 5. Vantagens, hipóteses e limitações

- Lida bem com preditores correlacionados e pode reduzir dimensionalidade de modo supervisionado.
- É um modelo linear no espaço transformado; não representa automaticamente limiares e interações não lineares complexas.
- Exige escala apropriada das variáveis; a configuração usada ativa `scale=True`.
- A interpretação dos componentes é menos direta que a leitura de uma regressão com poucas variáveis independentes.
- A escolha de componentes e todo o pré-processamento precisam ocorrer dentro de cada partição temporal, sem olhar o futuro.

## 6. Protocolo temporal deste projeto

A série é diária, com 1.996 linhas de treino (70%) e 856 linhas de teste (30%). O alvo `target` representa o fechamento do próximo dia; as features foram preparadas na etapa anterior. O número de componentes foi escolhido somente no treino por `TimeSeriesSplit`; a avaliação principal reportada é o walk-forward fora da amostra. No teste, os hiperparâmetros ficam fixos.

## 7. Interpretação do PLS

- **Coeficientes:** direção e magnitude da relação condicional do modelo; precisam ser lidos considerando o escalonamento e a correlação entre features.
- **VIP (Variable Importance in Projection):** resume a contribuição das variáveis para os componentes relevantes à resposta; VIP acima de 1 costuma ser usado como heurística, não como teste de significância.
- **Permutation importance:** mede quanto piora uma métrica quando uma feature é embaralhada. Deve ser interpretada como importância preditiva, não causal; features correlacionadas podem dividir ou mascarar importância.

## Referências

- Trabalho de Séries Temporais, seção 9: estudo técnico do modelo de especialização.
- scikit-learn, `PLSRegression`: https://scikit-learn.org/stable/modules/generated/sklearn.cross_decomposition.PLSRegression.html
- Wold, Sjöström e Eriksson (2001), *PLS-regression: a basic tool of chemometrics*.